In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('CoralBleachingBigData').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')  # reduce log noise

df = spark.read.csv('../data/raw/archive 2/coral.csv', header=True, inferSchema=True)

print("Total rows:", df.count())
print("Total columns:", len(df.columns))
df.printSchema()

C:\Users\Asthi\coral-reef-health-ai\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Total rows: 41361
Total columns: 62
root
 |-- Site_ID: integer (nullable = true)
 |-- Sample_ID: integer (nullable = true)
 |-- Data_Source: string (nullable = true)
 |-- Latitude_Degrees: double (nullable = true)
 |-- Longitude_Degrees: double (nullable = true)
 |-- Ocean_Name: string (nullable = true)
 |-- Reef_ID: string (nullable = true)
 |-- Realm_Name: string (nullable = true)
 |-- Ecoregion_Name: string (nullable = true)
 |-- Country_Name: string (nullable = true)
 |-- State_Island_Province_Name: string (nullable = true)
 |-- City_Town_Name: string (nullable = true)
 |-- Site_Name: string (nullable = true)
 |-- Distance_to_Shore: string (nullable = true)
 |-- Exposure: string (nullable = true)
 |-- Turbidity: string (nullable = true)
 |-- Cyclone_Frequency: double (nullable = true)
 |-- Date_Day: integer (nullable = true)
 |-- Date_Month: integer (nullable = true)
 |-- Date_Year: integer (nullable = true)
 |-- Depth_m: string (nullable = true)
 |-- Substrate_Name: string (nullab

In [2]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType

# Columns that should be numeric but are stored as string due to "nd" values
numeric_cols = ['Distance_to_Shore', 'Turbidity', 'Depth_m', 'Percent_Cover',
                 'Percent_Bleaching', 'ClimSST', 'Temperature_Kelvin', 'Temperature_Mean',
                 'Temperature_Minimum', 'Temperature_Maximum', 'Temperature_Kelvin_Standard_Deviation',
                 'Windspeed', 'SSTA', 'SSTA_Standard_Deviation', 'SSTA_Mean', 'SSTA_Minimum',
                 'SSTA_Maximum', 'SSTA_Frequency', 'SSTA_Frequency_Standard_Deviation',
                 'SSTA_FrequencyMax', 'SSTA_FrequencyMean', 'SSTA_DHW', 'SSTA_DHW_Standard_Deviation',
                 'SSTA_DHWMax', 'SSTA_DHWMean', 'TSA', 'TSA_Standard_Deviation', 'TSA_Minimum',
                 'TSA_Maximum', 'TSA_Mean', 'TSA_Frequency', 'TSA_Frequency_Standard_Deviation',
                 'TSA_FrequencyMax', 'TSA_FrequencyMean', 'TSA_DHW', 'TSA_DHW_Standard_Deviation',
                 'TSA_DHWMax', 'TSA_DHWMean']

for c in numeric_cols:
    df = df.withColumn(c, when(col(c) == 'nd', None).otherwise(col(c)).cast(DoubleType()))

# Drop rows where our key target column is missing
df_clean = df.filter(col('Percent_Bleaching').isNotNull())

print("Rows after cleaning:", df_clean.count())

Rows after cleaning: 34515


In [3]:
from pyspark.sql.functions import avg, count, stddev, max as spark_max, min as spark_min

# Aggregate: average bleaching and temperature metrics by Ocean + Year
regional_yearly_trends = df_clean.groupBy('Ocean_Name', 'Date_Year').agg(
    avg('Percent_Bleaching').alias('avg_bleaching_pct'),
    count('*').alias('num_observations'),
    avg('Temperature_Mean').alias('avg_temperature'),
    avg('SSTA').alias('avg_ssta'),
    avg('TSA').alias('avg_tsa'),
    spark_max('Percent_Bleaching').alias('max_bleaching_pct')
).orderBy('Ocean_Name', 'Date_Year')

regional_yearly_trends.show(20)

+------------+---------+-------------------+----------------+------------------+--------------------+--------------------+-----------------+
|  Ocean_Name|Date_Year|  avg_bleaching_pct|num_observations|   avg_temperature|            avg_ssta|             avg_tsa|max_bleaching_pct|
+------------+---------+-------------------+----------------+------------------+--------------------+--------------------+-----------------+
|Arabian Gulf|     1996|  78.53846153846153|              13| 299.7438461538461|  0.3823076923076923| -0.7576923076923078|            100.0|
|Arabian Gulf|     1998| 54.611111111111114|               9|            299.62|  3.1611111111111114|   2.878888888888889|             75.0|
|Arabian Gulf|     2000| 60.166666666666664|               3|298.68666666666667| 0.01666666666666668| -0.7433333333333333|             75.0|
|Arabian Gulf|     2002|               75.0|               1|            300.93|                1.22|                1.23|             75.0|
|Arabian Gulf

In [5]:
# Convert the aggregated Spark result to pandas and save (avoids Windows Hadoop/winutils issue)
regional_yearly_trends_pd = regional_yearly_trends.toPandas()
regional_yearly_trends_pd.to_csv('../data/processed/regional_yearly_trends.csv', index=False)

print("Saved regional_yearly_trends.csv")
print("Shape:", regional_yearly_trends_pd.shape)

C:\Users\Asthi\coral-reef-health-ai\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Asthi\coral-reef-health-ai\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Saved regional_yearly_trends.csv
Shape: (131, 8)


In [6]:
from pyspark.sql.functions import when as spark_when, count

country_severity = df_clean.withColumn(
    'severity',
    spark_when(col('Percent_Bleaching') == 0, 'None')
    .when(col('Percent_Bleaching') <= 10, 'Low')
    .when(col('Percent_Bleaching') <= 50, 'Moderate')
    .otherwise('Severe')
).groupBy('Country_Name', 'severity').agg(
    count('*').alias('count')
).orderBy('Country_Name', 'severity')

country_severity.show(20)

# Save this too
country_severity_pd = country_severity.toPandas()
country_severity_pd.to_csv('../data/processed/country_severity_breakdown.csv', index=False)
print("Saved country_severity_breakdown.csv, shape:", country_severity_pd.shape)

+-------------------+--------+-----+
|       Country_Name|severity|count|
+-------------------+--------+-----+
|Antigua and Barbuda|     Low|    2|
|Antigua and Barbuda|Moderate|    2|
|Antigua and Barbuda|    None|    2|
|          Australia|     Low| 1398|
|          Australia|Moderate|  364|
|          Australia|    None|  477|
|          Australia|  Severe|  384|
|            Bahamas|     Low|  292|
|            Bahamas|Moderate|   95|
|            Bahamas|    None|  258|
|            Bahamas|  Severe|   34|
|            Bahrain|Moderate|    1|
|            Bahrain|  Severe|   11|
|         Bangladesh|     Low|    4|
|           Barbados|     Low|   19|
|           Barbados|Moderate|   17|
|           Barbados|    None|   18|
|           Barbados|  Severe|    9|
|             Belize|     Low|  420|
|             Belize|Moderate|  102|
+-------------------+--------+-----+
only showing top 20 rows


C:\Users\Asthi\coral-reef-health-ai\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Asthi\coral-reef-health-ai\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Saved country_severity_breakdown.csv, shape: (289, 3)
